# 监控与可观测性 第1周:Prometheus — 指标采集与存储

> **学习目标**:理解可观测性三支柱,掌握 Prometheus 数据模型和 PromQL,能为 Python 应用添加自定义 metrics

---

## Day 1:可观测性三支柱

### 为什么三个支柱缺一不可？

| 支柱 | 回答的问题 | 数据形式 | 典型工具 |
|------|-----------|---------|---------|
| **Metrics(指标)** | "有没有问题？" | 数值型时间序列 | Prometheus, VictoriaMetrics |
| **Logs(日志)** | "具体是什么问题？" | 不可变事件记录 | Loki, ELK |
| **Traces(追踪)** | "问题出在哪个环节？" | 请求的完整调用路径 | Jaeger, Tempo |

### Google 四大黄金信号

1. **Latency(延迟)** — 请求花了多长时间？关注 p50/p95/p99
2. **Traffic(流量)** — 系统承受了多少请求？QPS/TPS
3. **Errors(错误)** — 多少请求失败了？按错误类型细分
4. **Saturation(饱和度)** — 资源利用率怎么样了？CPU/内存/连接池/队列深度

In [ ]:
print("为 Web API 定义服务水平指标(SLI)和 服务等级目标(SLO)")
print("=" * 50)

slis = [
    ("延迟 (Latency)", "GET /api 的 p99 延迟", "p99 < 200ms"),
    ("可用性 (Availability)", "非 5xx 响应占比", "> 99.9% (每月 downtime < 43min)"),
    ("吞吐量 (Throughput)", "每秒总请求数", "N/A (仅监控)"),
    ("错误率 (Error Rate)", "5xx 错误比例", "< 0.1%"),
]

print(f"{'SLI':<15} {'SLO'}")
for name, sli, slo in slis:
    print(f"{name:<15} {slo}")

print()
print("Error Budget = 1 - SLO")
print("如果 SLO 是 99.9% 可用 -> Error Budget = 0.1%")
print("每月 error budget 用完了 -> 停止发布新功能,专注修复稳定性")

## Day 2:Prometheus 架构与数据模型

### Pull 模型 vs Push 模型

```
传统监控 (Push):  Agent → push → 监控服务器
Prometheus (Pull): Prometheus → scrape → /metrics 端点

Pull 模型的优势:
- 不需要在目标上安装额外 agent(HTTP /metrics 就够了)
- 可以自动发现新的 target(K8s Service Discovery)
- 每个 target 的 /metrics 是一个快照,随时可用浏览器查看
```

### 四种指标类型

In [ ]:
print("Prometheus 四种指标类型")
print("=" * 50)

types = {
    "Counter": {
        "特点": "只增不减(除非重置),必须用 rate() 查询",
        "示例": "http_requests_total, errors_total",
        "关键函数": "rate(), increase()",
    },
    "Gauge": {
        "特点": "可增可减,直接取值即可",
        "示例": "cpu_usage_percent, memory_bytes, queue_size",
        "关键函数": "avg_over_time(), max_over_time()",
    },
    "Histogram": {
        "特点": "分桶统计,记录了总数+各桶的计数(推荐)",
        "示例": "http_request_duration_seconds",
        "关键函数": "histogram_quantile(0.99, rate(...[5m]))",
    },
    "Summary": {
        "特点": "客户端直接计算分位数,无法跨实例聚合(不推荐)",
        "示例": "go_gc_duration_seconds",
        "注意": "尽量用 Histogram 代替 Summary",
    },
}

for name, info in types.items():
    print(f"\n{name}: {info['特点']}")
    for k, v in info.items():
        if k != "特点":
            print(f"  {k}: {v}")

## Day 3:PromQL 入门

In [ ]:
print("PromQL 常用查询模式")
print("=" * 60)

queries = [
    ("当前值", "http_requests_total", "即时查询,返回最近一次 scrape 的值"),
    ("5分钟窗口", "http_requests_total[5m]", "范围向量,返回 5 分钟内所有数据点"),
    ("每秒速率", "rate(http_requests_total[5m])", "Counter 专用:计算每秒增长率"),
    ("5分钟增量", "increase(http_requests_total[5m])", "Counter 专用:过去 5 分钟增长了多少"),
    ("按标签聚合", "sum(rate(http_requests_total[5m])) by (endpoint)", "按 endpoint 分组求和"),
    ("p99 延迟", "histogram_quantile(0.99, rate(http_request_duration_seconds_bucket[5m]))", "从 Histogram 计算分位数"),
    ("错误率占比", 'sum(rate(http_requests_total{status_code=~"5.."}[5m])) / sum(rate(http_requests_total[5m]))', "5xx / 总量"),
    ("CPU 使用率", '100 - avg by(instance)(rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100', "1 - idle 比率"),
]

for name, query, note in queries:
    print(f"\n{name}:")
    print(f"  {query}")
    print(f"  → {note}")

## Day 4:Node Exporter — 主机指标

Node Exporter 暴露 Linux 主机指标:

| 类别 | 关键指标 | 说明 |
|------|---------|------|
| CPU | `node_cpu_seconds_total{mode="idle"}` | CPU 空闲时间 |
| Memory | `node_memory_MemAvailable_bytes` | 可用内存 |
| Disk | `node_filesystem_avail_bytes` | 磁盘可用空间 |
| Network | `node_network_receive_bytes_total` | 网络接收字节 |

## Day 5:应用指标 — Python 集成

In [ ]:
print("prometheus_client 核心模式")
print("=" * 50)

print("""
from prometheus_client import Counter, Histogram, Gauge, start_http_server

# Counter: 记录 API 调用次数
REQUEST_COUNT = Counter(
    'app_requests_total', 'Total requests',
    ['method', 'endpoint', 'status']
)

# Histogram: 记录请求处理时间
REQUEST_DURATION = Histogram(
    'app_request_duration_seconds', 'Request duration',
    ['method', 'endpoint'],
    buckets=[0.01, 0.05, 0.1, 0.25, 0.5, 1, 2.5, 5, 10]
)

# 在 middleware 中使用:
# REQUEST_COUNT.labels(method='GET', endpoint='/api', status='200').inc()
# REQUEST_DURATION.labels(method='GET', endpoint='/api').observe(0.045)

# 暴露 /metrics 端点
start_http_server(8000)  # Prometheus scrape http://host:8000/metrics
""")

print("最佳实践:")
print("  1. 使用有意义的 metric 名称: app_requests_total 而不是 req_cnt")
print("  2. label 值不要高基数: 不要用 user_id / request_id 做 label")
print("  3. Counter 永远只增: 用 _total 后缀")
print("  4. Histogram bucket 根据 SLO 设定: 第一个桶覆盖正常延迟")

## Day 6:服务发现

| 方式 | 配置 | 适用 |
|------|------|------|
| static_configs | 写死 IP:Port | 固定 IP 的少量 target |
| file_sd_configs | JSON/YAML 文件自动加载 | 中小规模,自定义服务注册 |
| kubernetes_sd_configs | K8s API 自动发现 | K8s 环境的标准做法 |

## Day 7:第1周综合练习

In [ ]:
print("=" * 60)
print("第1周综合练习:搭建指标采集链路")
print("=" * 60)

print("""
目标:用 Docker Compose 搭建完整的指标采集链路

docker-compose.yml:
  ├── Prometheus (scrape 中心,端口 9090)
  ├── Node Exporter (主机指标,端口 9100)
  └── Python Web App (带 /metrics,端口 8000)

验证:
  1. curl localhost:9090/targets      # 看 target 状态
  2. curl localhost:8000/metrics      # 看 Python 的 metrics
  3. 在 Prometheus UI 中查询:
     - rate(python_gc_collections_total[5m])
     - app_requests_total
     - histogram_quantile(0.99, rate(app_request_duration_seconds_bucket[5m]))
""")

print("=" * 60)
print("第1周核心收获:")
print("1. Metrics 告诉你有没有问题,Logs 告诉是什么问题,Traces 告诉问题在哪")
print("2. Prometheus 用 Pull 模型,从 /metrics 端点 scrape 数据")
print("3. 四种指标类型:Counter(rate) / Gauge(直接看) / Histogram(分位) / Summary(客户端分位)")
print("4. PromQL: rate() 给 Counter,histogram_quantile() 给 Histogram")
print("5. 为 Python 应用加 metrics 只需要引入 prometheus_client 库")
print("=" * 60)